# Convex Quadratic Knapsack Problem using Frank Wolfe (CQKP_FW)

In [1]:
import numpy as np
from numpy.linalg import eigh
from scipy.sparse import diags
from scipy.linalg import eigvalsh
import cvxpy as cp

In [142]:
def gen_CQKP(n, actv=0.5, rank=1.1, ecc=0.99, seed=0, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True):
    # Generate a random instance of the Convex Quadratic Knapsack Problem (CQKP)
    if not isinstance(n, int) or n <= 0:
        raise ValueError("n must be a positive integer")
    if not (0 <= actv <= 1):
        raise ValueError("actv must be in [0, 1]")
    if rank <= 0:
        raise ValueError("rank must be > 0")
    if not (0 <= ecc < 1):
        raise ValueError("ecc must be in [0, 1)")
    if lmin < 0:
        raise ValueError("lmin must be >= 0")
    if lmax <= lmin:
        raise ValueError("lmax must be > lmin")
    if lmax > umin:
        raise ValueError("lmax must be < umin")
    if umax <= umin:
        raise ValueError("umax must be > umin")
    
    np.random.seed(seed)
    
    # Generate u
    l = lmin + (lmax - lmin) * np.random.rand(n)
    u = umin + (umax - umin) * np.random.rand(n)

    # Generate Q
    G = np.random.rand(round(rank * n), n)
    Q = G.T @ G
    
    # Assume Q is given and ecc (eccentricity) is given
    # Compute eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(Q)  # Sorted by increasing eigenvalue

    # Check if the smallest eigenvalue is greater than "0"
    if eigenvalues[0] > 1e-14:
        # Modify eccentricity only if lambda_n > 0, for when lambda_n = 0 the
        # eccentricity is 1 by default

        lambda_n = eigenvalues[0] # Smallest eigenvalue
        lambda_1 = eigenvalues[-1] # Largest eigenvalue

        # Calculate new eigenvalues with the desired eccentricity
        new_eigenvalues = lambda_n + ((eigenvalues - lambda_n) / (lambda_1 - lambda_n)) * (2 * ecc / (1 - ecc)) * lambda_n
        # lambd = lambda_n * np.ones(n) + (lambda_n / (lambda_1 - lambda_n)) * \
        #         (2 * ecc / (1 - ecc)) * \
        #         ((eigenvalues - lambda_n)  * np.ones(n))
        # Construct the diagonal matrix of the new eigenvalues
        D_new = np.diag(new_eigenvalues)

        # Recompute Q with modified eccentricity
        Q = eigenvectors @ D_new @ eigenvectors.T
    
    # Generate q
    z = np.zeros(n)
    is_out_of_bound = np.random.rand(n) <= actv
    is_out_lr = np.random.rand(n) <= 0.5
    is_out_left = is_out_of_bound & is_out_lr
    is_out_right = is_out_of_bound & ~is_out_lr
    z[is_out_left] = - np.random.rand(sum(is_out_left)) * l[is_out_left]
    z[is_out_right] = u[is_out_right] * (1 + np.random.rand(sum(is_out_right)))
    z[is_out_of_bound] = np.random.rand(sum(is_out_of_bound)) * u[is_out_of_bound]
    
    q = -Q @ z

    # Generate a and b
    a = np.random.rand(n) + 1e-6
    if cut_through_box:
        # Choose 'b' s.t. the hyperplane a^T x = b cuts through the box [l, u].
        b = np.dot(a, (l + u) / 2)  # This ensures the hyperplane passes through the center of the box
    else:
        b = np.random.rand() + 1e-6
    
    
    return {"Q": Q, "q": q, "l": l, "u": u, "a": a, "b": b}

In [98]:
p = gen_CQKP(10, actv=0.5, rank=1.1, ecc=0.99, seed=0, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True)
p

{'Q': array([[0.94475897, 0.75001258, 0.95861539, 0.79029967, 0.56033056,
         0.56632064, 0.55422351, 0.55296073, 0.72487981, 0.58552698],
        [0.75001258, 0.86404293, 0.80647471, 0.7247639 , 0.53922693,
         0.58302944, 0.64840966, 0.55369333, 0.87236988, 0.6137945 ],
        [0.95861539, 0.80647471, 1.27596885, 0.74407745, 0.67331683,
         0.67063433, 0.74878203, 0.56468229, 0.89600576, 0.64498978],
        [0.79029967, 0.7247639 , 0.74407745, 0.92186892, 0.47619655,
         0.56391525, 0.58549794, 0.5268874 , 0.76465337, 0.6689782 ],
        [0.56033056, 0.53922693, 0.67331683, 0.47619655, 0.59768277,
         0.38137952, 0.48132846, 0.28015438, 0.57109251, 0.45725073],
        [0.56632064, 0.58302944, 0.67063433, 0.56391525, 0.38137952,
         0.59023301, 0.4727686 , 0.46001035, 0.64694314, 0.53928533],
        [0.55422351, 0.64840966, 0.74878203, 0.58549794, 0.48132846,
         0.4727686 , 0.82917347, 0.41507164, 0.83737704, 0.59602141],
        [0.55296073, 0

In [99]:
def is_positive_semidefinite(Q):
    """
    Check if a matrix Q is positive semidefinite (PSD).
    
    Args:
    - Q (numpy.ndarray): A square numpy matrix.
    
    Returns:
    - bool: True if Q is PSD, False otherwise.
    """
    # Check if the matrix is symmetric, as all PSD matrices must be.
    if not np.allclose(Q, Q.T, atol=1e-8):
        return False
    
    # Compute eigenvalues
    eigenvalues = np.linalg.eigvalsh(Q)
    
    # Check if all eigenvalues are non-negative
    return np.all(eigenvalues >= -1e-8)

print("Is Q positive semidefinite?", is_positive_semidefinite(p["Q"]))

Is Q positive semidefinite? True


In [100]:
def linear_subproblem(c, a, b, l, u, verbose=False):
    n = len(c)
    x_star = []
    for i in range(n):
        if c[i] >= 0:
            x_star.append(l[i])
        else:
            x_star.append(u[i])
    
    if (a @ x_star) >= b:
        if verbose:
            print("Feasible solution in a corner of the hyperbox")
        return x_star
    
    # The minimum value of the objective function is not in feasible region
    x_pack = [(i, x_star[i], c[i], a[i]) for i in range(n)]
    # Order in ascending order of c_i/a_i
    x_pack = sorted(x_pack, key=lambda x: x[2] / x[3], reverse=False)
    ordered_index = [t[0] for t in x_pack]
    
    for idx in ordered_index:
        x_star[idx] = u[idx]
        
        if (a @ x_star) > b:
            if verbose:
                print("Feasible solution NOT in a corner of the hyperbox")
            x_star[idx] = (b - (a @ x_star) + (a[idx] * x_star[idx])) / a[idx]
            return x_star
        
        
    if verbose:
        print("No feasible solution found")
    return x_star

In [135]:
def CQKP_FW(CQKP, t=0, eps=1e-6, MaxIter=1000, verbose=True):
    """
    Apply the (possibly, stabilized) Franke-Wolfe algorithm with exact line
    search to the convex Box-Constrained Quadratic program
    
    (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }
    """
    # Initializations
    n = CQKP['q'].shape[0]
    x = CQKP['u'] / 2  # Start from the middle of the box
    best_lower_bound = -np.inf  # Best lower bound so far

    if verbose:
        print("Frank-Wolfe method")
        print("iter\tf(x)\t\tlower_bound\t\tgap\n")

    i = 1

    while True:
        # Compute function value and direction = - gradient
        v = 0.5 * np.dot(x.T, np.dot(CQKP['Q'], x)) + np.dot(CQKP['q'].T, x)
        g = np.dot(CQKP['Q'], x) + CQKP['q']

        # Solve min { <g, z> : az >= b, l <= z <= u }
        x_bar = linear_subproblem(g, CQKP['a'], CQKP['b'], CQKP['l'], CQKP['u'], verbose=False)

        # Compute the lower bound
        lower_bound = v + np.dot(g.T, x_bar - x)
        if lower_bound > best_lower_bound:
            best_lower_bound = lower_bound

        # Compute the relative gap
        gap = (v - best_lower_bound) / max(abs(v), 1)

        if verbose:
            print(f"{i}\t{v:.8e}\t{best_lower_bound:.8e}\t{gap:.4e}")

        # Stopping criteria
        if gap <= eps:
            status = 'optimal'
            break
        if i > MaxIter:
            status = 'stopped'
            break

        # In the stabilized case, restrict x_bar in the box
        if t > 0:
            x_bar = np.maximum(x - t * CQKP['u'], np.minimum(x + t * CQKP['u'], x_bar))

        # Compute step size
        d = x_bar - x
        den = np.dot(d.T, np.dot(CQKP['Q'], d))
        if den <= 1e-16:
            alpha = 1  # F is linear along d, take max possible stepsize
        else:
            alpha = min(-np.dot(g.T, d) / den, 1)

        # Compute new point
        x = x + alpha * d
        i += 1

    if verbose:
        print(f"stop: {i} iter, status = {status}, fbest = {v:.8e}, gap = {gap:.4e}")

    return v, x, status


In [144]:
problem = gen_CQKP(10, actv=0.5, rank=1.1, ecc=0.99, seed=0, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True)
CQKP_FW(problem, eps=1e-6, MaxIter=100, verbose=True)

Frank-Wolfe method
iter	f(x)		lower_bound		gap

1	-9.17201466e+01	-1.83196758e+02	9.9734e-01
2	-1.59098436e+02	-1.62000287e+02	1.8239e-02
3	-1.60154157e+02	-1.60154157e+02	0.0000e+00
stop: 3 iter, status = optimal, fbest = -1.60154157e+02, gap = 0.0000e+00


(-160.15415683745562,
 array([ 2.19525402,  2.86075747,  2.4110535 ,  2.17953273,  7.35047181,
         8.3485172 ,  6.23477008, 11.33047938,  3.85465104,  1.53376608]),
 'optimal')

In [145]:
# (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }

# Define variables
Q = problem['Q']    
q = problem['q']
a = problem['a']
b = problem['b']
l = problem['l']
u = problem['u']
n = len(q)

x = cp.Variable(n)

# Define constraints
constraints = [a @ x >= b, x >= l, x <= u]

# Define objective
objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

# Define problem
P = cp.Problem(objective, constraints)
P.solve()

# Print result
print(f"Optimal value: {P.value}")
print(f"Optimal x: {x.value}")
print(f"Status: {P.status}")

Optimal value: -160.15415683745562
Optimal x: [ 2.19525402  2.86075747  2.4110535   2.17953273  7.35047181  8.3485172
  6.23477008 11.33047938  3.85465104  1.53376608]
Status: optimal


In [154]:
tries = 50
failed = 0

for i in range(0, tries):
    random_bool = np.random.rand() < 0.5
    data = gen_CQKP(100, actv=0.5, rank=1.1, ecc=0.99, seed=i, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=random_bool)

    # Define variables
    Q = data['Q']
    q = data['q']
    a = data['a']
    b = data['b']
    l = data['l']
    u = data['u']
    n = len(q)

    x = cp.Variable(n)
    # Define constraints
    constraints = [a @ x >= b, x >= l, x <= u]

    # Define objective
    objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

    # Define problem
    P = cp.Problem(objective, constraints)
    P.solve()
    gold_solution = [P.value, x.value]
    y, x_opt, _ = CQKP_FW(data, verbose=False)
    our_solution = [y, x_opt]

    if not np.allclose(gold_solution[0], our_solution[0], rtol=1e-3):
        print(f"Test {i} failed")
        print(f"Gold solution: {gold_solution}")
        print(f"Our solution: {our_solution}")
        failed += 1

if failed == 0:
    print(f"All {tries} tests passed")
else:
    print(f"{failed} out of {tries} tests failed")
    

All 50 tests passed
